> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 03. ROS Image와 OpenCV 연결 / 60분

**선행:** OpenCV 이미지 shape와 01–02의 topic

**이 노트북의 결과:** Image memory layout·timestamp를 검사하고 단일 camera publisher를 여러 처리 node가 구독하는 구조를 실행합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

영상 메시지는 픽셀 bytes만이 아니라 그 bytes를 읽는 방법도 포함합니다. 높이·너비·encoding·step과 촬영 시각을 함께 확인해야 다른 node가 같은 영상으로 해석할 수 있습니다. 한 카메라를 한 node가 열고 다른 node는 구독합니다.

### 전체 과정에서 내 코드가 하는 일

```text
카메라 또는 녹화 publisher
  ↓
ROS Image
  ↓
layout·padding TODO
  ↓
활성 픽셀 평균
  ↓
촬영 age
  ↓
수신 결과 기록
```

행 padding을 실제 픽셀 계산에서 제외하고 신선도 경계를 검사합니다. 현재 수신 시각으로 대체한 timestamp는 실제 촬영 시각 확인과 다르다는 점을 기록합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 같은 해상도인데 data 길이가 W×H×채널보다 클 수 있나요?

**설명:** 행마다 padding이 있으면 더 클 수 있습니다. step으로 다음 행의 시작을 찾고 유효 픽셀만 계산합니다. padding을 밝기로 포함하면 값이 달라지므로 layout 검사와 신선도 검사를 따로 확인합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**ROS 준비 — 노트북의 Python 모의 실험과 실제 ROS 실행 구분:** 아래 Python 셀은 새 fixture와 loader를 직접 만들므로 앞 노트북의 커널 변수·TODO 결과가 필요 없습니다. 실제 node는 **같은 개인 컨테이너의 JupyterLab Terminal** 에서 실행합니다. Ollama는 필요 없습니다.

[01_ros_basics.ipynb의 3-1~3-3절](01_ros_basics.ipynb)에 workspace 빌드와 `source`의 설명이 있으며, 필요한 절차는 다음과 같습니다. 빌드는 `ros_ws/src`의 패키지를 `Data/build/08.ROS/install`에 설치·연결합니다. 첫 사용 또는 install 파일이 없을 때, 패키지 설정·실행 이름이 바뀌었을 때 다음을 실행하고 성공을 확인합니다. 같은 자료 폴더·호환 이미지의 빌드 결과가 남아 있으면 재부팅만을 이유로 다시 빌드할 필요는 없습니다.

```bash
bash
cd /workspace
bash 08.ROS/scripts/build_workspace.sh
```

**새 터미널 A·B·C마다** 아래를 적용합니다. 이 설정은 현재 터미널에만 적용되며 다른 터미널·Python 커널로 전파되지 않습니다. 21은 로컬 예시이고 원격 개인 실습은 22 등 배정값을 씁니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 pkg executables nepes_ros_lab
```

`status_talker`, `image_monitor`, `safe_turtle_controller` 등 실행 이름이 보이면 준비된 것입니다. 기본 ROS는 첫 source, 수업 패키지는 두 번째 source가 찾게 합니다. `Package not found`는 build 성공·현재 install source를 확인합니다. 같은 실습의 모든 터미널은 같은 Domain 번호를 사용합니다. 로컬 GUI를 함께 관찰할 때는 두 수강생이 로컬 컨테이너의 화면을 함께 보고, 원격 개인 Domain에서 로컬 토픽이 보일 것으로 기대하지 않습니다. 이전 실습의 source/controller·카메라를 종료한 뒤 다음 실행을 시작합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[06.OpenCV/01_image_video_io.ipynb의 1절](../06.OpenCV/01_image_video_io.ipynb)은 BGR uint8 `(H,W,3)` 배열과 None 검사를, [02_turtlesim_tools.ipynb의 1절](02_turtlesim_tools.ipynb)은 node·topic·type·endpoint 관찰을 설명합니다. 여기서 필요한 내용은 **카메라는 한 publisher만 열고 처리기들은 같은 Image topic을 구독**, `cv_bridge`는 Image↔배열 변환이라는 점입니다. Image의 width/height·encoding·step·data·captured_ms는 아래 fixture에서 새로 만들며 앞 노트북 이미지 산출물은 필요 없습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–4절의 수업 핵심 경로** → [3일차 슬라이드 58쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s054) · [배포 PDF 58쪽](../슬라이드/3일차.pdf#page=58)에서 66쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 입력 토픽·처리 결과·출력 명령을 확인하고 종료합니다.
- **단원 마지막 연결 질문:** 입력 노드를 중단하고 마지막 관측 시각, STOP, 복구 후 새 관측을 확인합니다. 운영자가 어느 로그부터 볼지 설명합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

[ROS 터미널 운영표](../08.ROS/실행안내.md)를 사용하세요. 모의 입력은 이동·정지를, 실제 카메라는 인식·승인 또는 거부 이유를 확인합니다. 

08의 MediaPipe confidence=null 이동은 기본 controller에서 거부됩니다. 입력 중단 실험은 실행안내의 source·controller 분리 명령으로 진행합니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 시간 지도

카메라 소유권 8분 → Image 필드 12분 → byte 예제 10분 → TODO 15분 → v4l2/cv_bridge 10분 → 종료 5분.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "08.ROS").is_dir():
            return candidate
    raise RuntimeError("08.ROS를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "08.ROS"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('rclpy', 'cv2', 'cv_bridge')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 입력 장치를 한 node만 연다

```text
/dev/video0 → v4l2_camera → /camera/image_raw
                              ├ OpenCV
                              ├ MediaPipe
                              ├ YOLO/VLM
                              └ rqt_image_view(local)
```

각 처리기가 장치를 직접 열면 점유 충돌과 frame 시각 불일치가 생깁니다. `sensor_msgs/Image`의 width/height, encoding, step, data와 header timestamp를 함께 전달합니다.

`mono8`은 pixel당 1 byte, `rgb8/bgr8`은 3 byte입니다. 활성 row는 `width×channels`이고 `step`은 padding 때문에 더 클 수 있습니다. Data 길이는 `step×height`입니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.contracts import ImageMessage
FRAME = ImageMessage(
    width=2, height=2, encoding="mono8", step=3,
    data=bytes([10, 20, 255, 30, 40, 255]),
    captured_ms=1000, frame_id="frame-001",
)
print("각 행 끝의 255는 padding입니다. 활성 pixel 평균을 실행 전에 계산하세요.")
print({"active_bytes": 4, "stored_bytes": len(FRAME.data), "expected_mean": 25.0})


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. TODO — Layout와 freshness

[exercises/image_contract.py](exercises/image_contract.py)의 `mean_active_pixels`를 구현합니다. 크기, 지원 encoding, 최소 step, 정확한 data 길이와 `0 ≤ now-captured ≤ 500 ms`를 검사한 뒤 각 행의 padding을 제외한 평균을 냅니다. 음수 age도 future/clock 오류라서 거부합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `mean_active_pixels(message,now_ms=...,max_age_ms=500)`는 제공 `ImageMessage`와 밀리초 시각을 받아 float 평균을 반환합니다. mono8 W=2,H=2,step=3,data=`bytes([10,20,255,30,40,255])`이면 평균은 `25.0`입니다.
**작성 흐름 힌트:** 크기·encoding 검사 → channels(mono8=1,rgb8/bgr8=3)로 active=width×channels → step≥active와 정확한 data 길이 검사 → 시각 정수·age 경계 검사 → 행마다 start=row×step에서 active byte만 누적 → 활성 byte 수로 나눔.
**주의할 경계:** RGB에서도 각 채널 byte 전체 평균이지 grayscale 변환이 아닙니다. 500ms는 허용, 501ms·미래 프레임은 TimeoutError입니다. padding 255를 평균에 포함하지 않습니다.
**설명할 지점:** “shape뿐 아니라 한 행의 실제 byte 간격 step을 알아야 다음 행 시작을 찾습니다.”
**검사·예외를 정리하면:** `from common.contracts import ImageMessage`로 제공 자료형을 읽고 message가 그 타입이 아니면 `TypeError`를 냅니다. width·height는 양수, 지원 encoding·충분한 step·정확한 data 길이 위반은 `ValueError`입니다. captured_ms·now_ms·max_age_ms는 bool 제외 정수여야 하고 captured_ms·max_age_ms는 0 이상입니다. age가 음수 또는 max_age_ms보다 크면 `TimeoutError`; 정확한 양끝 0·max_age_ms는 허용합니다. 반환은 전체 활성 byte 합÷(width×channels×height)이며 새 이미지·ROS 메시지를 만드는 과제가 아닙니다. 아래 학습자 검증은 mono8/rgb8 평균, padding, 크기·길이·시각 오류를 모두 확인합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_ros_image", "exercises/image_contract.py")
assert student.mean_active_pixels(FRAME, now_ms=1500) == 25.0
rgb = ImageMessage(1, 1, "rgb8", 4, bytes([10, 20, 30, 255]), 1000, "rgb-padding")
assert student.mean_active_pixels(rgb, now_ms=1000) == 20.0

bad_cases = [
    (ImageMessage(0, 1, "mono8", 1, b"\x01", 1000, "zero-width"), 1000, 500, (TypeError, ValueError)),
    (ImageMessage(1, 0, "mono8", 1, b"", 1000, "zero-height"), 1000, 500, (TypeError, ValueError)),
    (ImageMessage(1, 1, "rgba8", 4, b"1234", 1000, "encoding"), 1000, 500, (TypeError, ValueError)),
    (ImageMessage(2, 1, "rgb8", 5, b"12345", 1000, "bad-step"), 1000, 500, (TypeError, ValueError)),
    (ImageMessage(1, 1, "mono8", 1, b"", 1000, "short-data"), 1000, 500, (TypeError, ValueError)),
    (ImageMessage(1, 1, "mono8", 1, b"\x01\x02", 1000, "long-data"), 1000, 500, (TypeError, ValueError)),
    (ImageMessage(1, 1, "mono8", 1, b"\x01", 1000, "stale"), 1501, 500, (TimeoutError,)),
    (ImageMessage(1, 1, "mono8", 1, b"\x01", 1000, "future"), 999, 500, (TimeoutError,)),
    (ImageMessage(1, 1, "mono8", 1, b"\x01", 1000, "time-type"), True, 500, (TypeError, ValueError)),
    (ImageMessage(1, 1, "mono8", 1, b"\x01", 1000, "age-type"), 1000, True, (TypeError, ValueError)),
]
for value, now, max_age, expected in bad_cases:
    try:
        student.mean_active_pixels(value, now_ms=now, max_age_ms=max_age)
    except expected:
        pass
    else:
        raise AssertionError(f"잘못된 image를 거부해야 합니다: {value.frame_id}")
assert student.mean_active_pixels(
    ImageMessage(1, 1, "mono8", 1, b"\x09", 1000, "age-boundary"),
    now_ms=1500,
    max_age_ms=500,
) == 9.0
print("PASS: 지원 encoding, padding, 크기·step·data 길이, freshness 양쪽 경계")


 > **구분: 수업 핵심**

## 참고 정답

강사가 안내하는 시점에 아래 참고 정답과 해설을 확인합니다. [solutions/image_contract.py](solutions/image_contract.py)에 입력 검사·계산·반환이 직접 구현되어 있으며, 아래 접힌 Markdown에도 같은 코드가 있습니다. 정답 검증과 학습자 TODO 검증은 별도입니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_ros_image", "solutions/image_contract.py")
assert reference.mean_active_pixels(FRAME, now_ms=1500) == 25.0
print("PASS: 참고 정답 검증")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/image_contract.py](solutions/image_contract.py)의 `mean_active_pixels`입니다. encoding으로 채널 수를 정하고, 한 행의 유효 bytes는 width×channels입니다. step은 다음 행 시작까지의 거리이므로 뒤에 padding이 포함될 수 있습니다.

mono8·width=2·height=2·step=3, data=[10,20,255,30,40,255]라면 활성 값은 10·20·30·40이고 평균은 25입니다. 255 padding을 포함하면 영상에 없는 값으로 평균을 왜곡합니다.

age=now_ms-captured_ms를 계산하고 미래 시각이나 max_age_ms를 초과한 프레임은 거부합니다. 이 함수는 정확히 500ms인 기본 경계를 허용합니다. data 길이도 step×height와 맞아야 합니다.

**확인 질문:** 배열 길이가 맞아도 encoding을 mono8 대신 rgb8로 잘못 적으면 왜 픽셀 해석이 달라지나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/image_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: ImageMessage의 encoding·행 간격·바이트 길이·시각을 검사합니다.
각 행에서 padding을 제외한 활성 바이트만 모아 평균을 반환합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
# 공통 입력 자료형만 공유합니다. 평균 계산은 아래에 직접 구현합니다.
from common.contracts import ImageMessage

def mean_active_pixels(message: ImageMessage, *, now_ms: int, max_age_ms: int = 500) -> float:
    """행 padding을 제외한 활성 pixel byte 평균을 계산한다."""
    if not isinstance(message, ImageMessage):
        raise TypeError("message는 ImageMessage여야 합니다")
    if message.width <= 0 or message.height <= 0:
        raise ValueError("width와 height는 양수여야 합니다")
    channels = {"mono8": 1, "rgb8": 3, "bgr8": 3}.get(message.encoding)
    if channels is None:
        raise ValueError(f"지원하지 않는 encoding: {message.encoding}")
    active = message.width * channels
    if message.step < active:
        raise ValueError("step은 한 행의 활성 byte 수보다 작을 수 없습니다")
    if len(message.data) != message.step * message.height:
        raise ValueError("data 길이는 step × height와 같아야 합니다")
    if any(isinstance(v, bool) or not isinstance(v, int) for v in (message.captured_ms, now_ms, max_age_ms)):
        raise TypeError("captured_ms, now_ms, max_age_ms는 bool이 아닌 정수여야 합니다")
    if message.captured_ms < 0 or max_age_ms < 0:
        raise ValueError("captured_ms와 max_age_ms는 0 이상이어야 합니다")
    age = now_ms - message.captured_ms
    if age < 0 or age > max_age_ms:
        raise TimeoutError(f"frame age {age}ms가 허용 범위를 벗어났습니다")
    # 행 간격 step에는 padding이 포함되므로 활성 byte만 잘라 평균을 냅니다.
    values: list[int] = []
    for row in range(message.height):
        start = row * message.step
        values.extend(message.data[start : start + active])
    return sum(values) / len(values)
```

</details>
<!-- todo-review:answer:end -->


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 실제 ROS/OpenCV 경로

<!-- procedure:lesson-28-77a79d:start -->
![카메라 한 곳에서 열고 전달하기](../교재그림/절차/lesson-28.svg)

여러 처리 노드는 토픽을 구독합니다. 같은 장치를 각자 여는 구조와 구분하세요.
<!-- procedure:lesson-28-77a79d:end -->


```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
# A: camera를 여는 유일한 node
ros2 run v4l2_camera v4l2_camera_node --ros-args   -p video_device:=/dev/video0 -p image_size:="[640,480]" -r image_raw:=/camera/image_raw
```

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
# B
ros2 run nepes_ros_lab image_monitor
```

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 원격은 22 등 본인 배정값
# C: headless
mkdir -p /workspace/Data/outputs/08.ROS
ros2 topic hz /camera/image_raw
ros2 topic echo /camera/mean_brightness --once | tee /workspace/Data/outputs/08.ROS/03-mean-brightness.txt
# 로컬 GUI만
ros2 run rqt_image_view rqt_image_view /camera/image_raw
```

`cv_bridge`는 message를 BGR 배열로 변환할 뿐 camera를 열지 않습니다. topic hz는 충분한 표본을 본 뒤 Ctrl+C로 끝내고, 밝기 한 건은 Data 파일과 화면에 함께 남깁니다.

**카메라 없이 같은 ROS 경로를 독립 실행하는 선택 방법:** 원격 개인 컨테이너에서는 A의 카메라 명령 대신 아래를 사용하고, B·C에는 앞의 준비 블록과 **같은 개인 Domain 번호** 를 적용합니다. 정적 파일을 반복 발행하는 경로이며 실시간 USB 촬영 성능 검증이 아닙니다. USB publisher와 동시에 켜지 않습니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=22 # 본인 원격 배정값
ros2 run nepes_ros_lab recorded_image_publisher --ros-args \
  -p image_source:=/workspace/Data/images/07.Mediapipe/person_open_palm.png
```

제공 PNG는 07의 사전 배포 입력이며 07 노트북에서 만들어야 하는 출력이 아닙니다. A는 구독자가 생길 때까지 기다립니다. B의 image_monitor를 시작한 뒤 밝기·빈도를 관찰하고 A·B를 각각 Ctrl+C로 종료합니다.

**관측 시각의 예외:** `mediapipe_gesture.py`는 Image의 `header.stamp`가 0이면 처리 시점의 현재 시각으로 `observed_ms`를 채웁니다. 이 경우 TTL은 대체된 시각부터 계산하므로 실제 촬영 시각의 신선도는 확인하지 못합니다. 기본 `confidence=None` 경로의 이동 거부는 유지되지만, 고정 `confidence=1.0` 체험에서는 촬영 시각이 없는 입력도 최신 관측처럼 처리될 수 있습니다. 고정 confidence는 인식 정확도나 영상의 신선도를 보장하지 않습니다. 실제 수업 카메라에서 0 stamp가 발생하는지는 별도 관찰이 필요하며, 정상 stamp가 있는 영상 전체의 문제라는 뜻은 아닙니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 4. 완료·오류 점검

640×480 BGR의 최소 row byte가 1920인 이유를 말합니다. Camera 실패는 device 전달·점유, 색 오류는 RGB/BGR, 이름만 보이고 frame이 없으면 QoS와 `topic hz`를 확인합니다. 원격 GUI가 없는 것은 정상이며 topic과 저장 결과로 검증합니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** layout/freshness 검증·카메라 발행·밝기 수신

[3일차 슬라이드 58쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s054) · [배포 PDF 58쪽](../슬라이드/3일차.pdf#page=58)로 돌아가 **58–66쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습 — 영상 대역폭과 QoS를 숫자로 비교하기 / 약 40분

Raw 영상의 초당 byte 수는 width × height × channels × fps입니다. MB/s는 여기서 1,000,000으로 나눈 십진 단위입니다. 이 값은 pixel payload만 센 것이며 ROS serialization, message header와 network overhead는 포함하지 않습니다.

행동 판단용 subscriber는 몇 초 전 frame을 모두 받는 것보다 최근 frame을 빠르게 보는 일이 중요할 수 있습니다. 이때 best effort와 depth 1을 후보로 검토합니다. 원본을 빠짐없이 보관해야 하는 기록은 실시간 행동 subscriber에 맡기지 않고 별도 recorder와 저장 성능을 설계합니다.

진행 순서는 두 영상 규격의 MB/s 예측 → 계산 검증 → 목적별 전달 정책 선택 → 실제 topic 측정 → Data 기록입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
STUDY_STREAMS = [
    {"id": "vga_bgr30", "width": 640, "height": 480, "channels": 3, "fps": 30},
    {"id": "hd_bgr30", "width": 1280, "height": 720, "channels": 3, "fps": 30},
]
for stream in STUDY_STREAMS:
    print(stream)
print("공식: width × height × channels × fps ÷ 1_000_000 = MB/s")


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### TODO — 계산값과 처리 목적 입력

다음 셀에서 bandwidth의 None을 소수 MB/s 숫자로 바꾸세요. 계산기 사용은 괜찮지만 먼저 단위를 적어 보세요.

Policy 답은 다음 두 문자열 중 하나입니다.

- best_effort_depth_1: 최신 영상 위주의 실시간 행동 입력
- separate_recorder: 모든 원본을 보관해야 하는 기록 경로

Reliable 하나만 선택한다고 disk가 느린 상황의 무손실 기록이 자동 보장되는 것은 아닙니다.

**입력과 편집 위치:** `MY_BANDWIDTH_MBPS`의 두 None에는 640×480×3×30과 1280×720×3×30 bytes/s를 각각 1,000,000으로 나눈 **숫자** 를 넣습니다. `"27 MB/s"` 같은 단위 포함 문자열은 넣지 않습니다. `MY_STREAM_POLICIES`의 live_control은 최신 장면 반응, lossless_archive는 모든 원본 보관 목적입니다. 두 key는 그대로 두고 위 두 정책 문자열 중 하나를 넣습니다. 네 값의 수정 셀→검증 셀을 실행하여 계산 오차 1e-6 미만·정책 두 개 일치 PASS를 확인하고 `03-bandwidth-qos.json`에 기록합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
MY_BANDWIDTH_MBPS = {
    "vga_bgr30": None,  # TODO: 640 × 480 × 3 × 30 ÷ 1_000_000
    "hd_bgr30": None,   # TODO: 1280 × 720 × 3 × 30 ÷ 1_000_000
}
MY_STREAM_POLICIES = {
    "live_control": None,    # TODO: "best_effort_depth_1" 또는 "separate_recorder"
    "lossless_archive": None,  # TODO
}
print("대역폭 답:", MY_BANDWIDTH_MBPS)
print("정책 답:", MY_STREAM_POLICIES)


In [ ]:
# 구분: 수업 후 복습·자습 셀
def raw_stream_report(stream):
    bytes_per_frame = stream["width"] * stream["height"] * stream["channels"]
    bytes_per_second = bytes_per_frame * stream["fps"]
    return {
        **stream,
        "bytes_per_frame": bytes_per_frame,
        "mb_per_second": bytes_per_second / 1_000_000,
    }

BANDWIDTH_REPORT = [raw_stream_report(stream) for stream in STUDY_STREAMS]
EXPECTED_BANDWIDTH = {item["id"]: item["mb_per_second"] for item in BANDWIDTH_REPORT}
EXPECTED_STREAM_POLICIES = {
    "live_control": "best_effort_depth_1",
    "lossless_archive": "separate_recorder",
}
def bandwidth_validation_errors(bandwidth, policies):
    """검증과 저장에서 현재 답에 같은 타입·key·허용 오차 규칙을 적용합니다."""
    import math
    errors = []
    if not isinstance(bandwidth, dict) or set(bandwidth) != set(EXPECTED_BANDWIDTH):
        errors.append("대역폭의 두 key를 그대로 유지하세요.")
    else:
        for name, expected in EXPECTED_BANDWIDTH.items():
            actual = bandwidth[name]
            if isinstance(actual, bool) or not isinstance(actual, (int, float)):
                errors.append(f"{name}: bool·문자열이 아닌 숫자가 필요합니다.")
            elif not math.isfinite(actual) or not abs(actual - expected) < 1e-6:
                errors.append(f"{name}: 단위와 channel 수를 다시 확인하세요.")
    if not isinstance(policies, dict) or policies != EXPECTED_STREAM_POLICIES:
        errors.append("목적별 정책의 두 key와 값을 확인하세요.")
    return errors

bandwidth_errors = bandwidth_validation_errors(MY_BANDWIDTH_MBPS, MY_STREAM_POLICIES)
bandwidth_study_passed = not bandwidth_errors
if any(value is None for value in MY_BANDWIDTH_MBPS.values()) or any(
    value is None for value in MY_STREAM_POLICIES.values()
):
    print("TODO가 남았습니다:", bandwidth_errors)
else:
    assert bandwidth_study_passed, "; ".join(bandwidth_errors)
    print("PASS: 두 raw 영상 대역폭과 목적별 stream 정책을 구분했습니다.")
print(json.dumps(BANDWIDTH_REPORT, ensure_ascii=False, indent=2))


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 실제 topic을 headless로 측정

이 노트북 3절의 로컬 카메라 또는 선택 recorded_image_publisher가 `/camera/image_raw`을 발행하는 동안 별도 컨테이너 Terminal에서 실행합니다. 그 터미널에도 `bash`, 기본 ROS·수업 install의 두 `source`, 발행자와 같은 `ROS_DOMAIN_ID`를 먼저 적용합니다. 각 측정 명령은 종료해야 다음 명령으로 넘어갑니다.

**새 측정 터미널에서 첫 블록 전체를 실행합니다.** hz의 표본을 확인한 뒤 Ctrl+C로 종료합니다.

```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
export ROS_DOMAIN_ID=21 # 본인 배정값 또는 공동 관찰의 로컬 배정값
ros2 topic hz /camera/image_raw
```

**같은 Bash에서** 다음 블록을 실행하고 충분한 표본을 본 뒤 Ctrl+C로 종료합니다. 두 측정 명령을 한 번에 붙여 넣지 않습니다.

```bash
ros2 topic bw /camera/image_raw
```

**같은 Bash에서** 발행자·구독자와 QoS를 확인합니다.

```bash
ros2 topic info /camera/image_raw --verbose
```

hz는 실제 도착 빈도, bw는 관측한 전송량입니다. 계산값과 다르면 encoding, 실제 해상도·fps, padding, 전송 overhead를 확인합니다. 명령은 계속 측정하므로 충분한 표본을 본 뒤 Ctrl+C로 끝냅니다.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 예상 결과

- vga_bgr30: 한 장 921,600 bytes, 초당 27.648 MB
- hd_bgr30: 한 장 2,764,800 bytes, 초당 82.944 MB
- live_control: best_effort_depth_1
- lossless_archive: separate_recorder

해상도의 가로와 세로를 각각 두 배로 하면 pixel 수와 raw 대역폭은 네 배가 됩니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
bandwidth_output = ROOT / "Data/outputs/08.ROS/03-bandwidth-qos.json"
bandwidth_output.parent.mkdir(parents=True, exist_ok=True)
# 과거 PASS 변수 대신 저장 시점의 답을 같은 함수로 다시 평가합니다.
bandwidth_errors = bandwidth_validation_errors(MY_BANDWIDTH_MBPS, MY_STREAM_POLICIES)
bandwidth_record = {
    "exercise": "bandwidth_qos",
    "computed": BANDWIDTH_REPORT,
    "bandwidth_answers": MY_BANDWIDTH_MBPS,
    "policy_answers": MY_STREAM_POLICIES,
    "completed": not bandwidth_errors,
    "validation_errors": bandwidth_errors,
}
bandwidth_output.write_text(
    json.dumps(bandwidth_record, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert bandwidth_output.is_file()
print("saved:", bandwidth_output)
print("completed:", bandwidth_record["completed"])
